# Coverage zones

This notebook builds coverage zones of the example services on an intermodal `UrbanGraph`: every place is assigned
to the service it reaches at the lowest network cost within a budget, so zones do not overlap. It shows

- `get_graph_coverage` by distance (800 m) and by time (10 min);
- `get_stepped_graph_coverage` in bands of 2 minutes;
- the graph-free `get_radius_coverage`.

The method is described on the [coverage page](../coverage.rst). The interactive maps are not stored in the notebook
because of their size; the figures on that page show the same results.

In [1]:
# Install required packages (uncomment if needed)
# !pip install iduedu pyarrow objectnat

In [2]:
import geopandas as gpd
from iduedu import config, download_osm_pbf, get_4326_boundary, get_intermodal_graph

from objectnat import get_graph_coverage, get_radius_coverage, get_stepped_graph_coverage

## 1. Intermodal graph

In [3]:
# Vasilyevsky Island district, Saint Petersburg (OSM relation 1114252).
# IduEdu >= 2.4 cuts the territory out of the Geofabrik extract (pip install "iduedu[pbf]");
# without download_osm_pbf, the calls below without the `with` block query the Overpass API.
pbf = download_osm_pbf(osm_id=1114252, region="northwestern-fed-district")
with config.use_pbf(pbf):
    poly = get_4326_boundary(osm_id=1114252)
    G_intermodal = get_intermodal_graph(territory=poly, clip_by_territory=True)

2026-11-04 22:06:48.953 | WARNING  | iduedu.graph.transformers:keep_largest_connected_component:64 - Removing 338 nodes outside the largest strongly connected component. Retaining 21181 of 21519 nodes.


2026-11-04 22:06:48.975 | WARNING  | iduedu.graph_builders.intermodal_builders:join_pt_walk_graph:248 - Public transport nodes dropped with the smaller components: {'trolleybus': 2}


## 2. Services

36 services with a `name` and a `capacity` column (EPSG:32636).

In [4]:
services = gpd.read_parquet('examples_data/services.parquet')

## 3. Coverage by distance

Zones within 800 m of network distance (`weight_type="length_meter"`).

In [5]:
# Calculate coverage zones by distance (800m cutoff)
result_length = get_graph_coverage(
    G_intermodal,
    gdf_destinations=services,
    weight_type="length_meter",
    weight_value_cutoff=800,
)

# Coverage results are indexed like the input services. Join service attributes
# back by index before coloring or filtering by service columns such as "name".
result_length = result_length.join(services[['name']], how="left")

# Visualize results
result_length.explore(column='name', tiles='CartoDB Positron')


## 4. Coverage by time

Zones within 10 minutes (`weight_type="time_min"`), clipped to the district boundary with `zone`.

In [6]:
# Prepare zone boundary
zone = gpd.GeoDataFrame(geometry=[poly], crs=4326)

# Calculate coverage zones by time (10min cutoff)
result_time = get_graph_coverage(
    G_intermodal,
    gdf_destinations=services,
    weight_type="time_min",
    weight_value_cutoff=10,
    zone=zone,
)

# Coverage results preserve the input service index, so service attributes can
# be joined back by index for visualization.
result_time = result_time.join(services[['name']], how="left")

# Visualize results
result_time.explore(column='name', tiles='CartoDB Positron')


### Parameters used

- `weight_type`: `"length_meter"` (metres) or `"time_min"` (minutes);
- `weight_value_cutoff`: the budget, in the units of `weight_type`;
- `zone` (optional): polygon the zones are clipped to; without it they are clipped to the concave hull of the
  reachable nodes;
- `geometry_type` (optional): `"radius"` or `"ways"` additionally clips the zones to residual-distance buffers or to
  the buffered streets.

## 5. Stepped coverage by time

The covered area split into bands of 2 minutes up to 15 minutes. `geometry_type=None` uses Voronoi cells of the
graph nodes; `"separate"` builds independent buffers per band.

In [7]:
stepped_cov_voronoi = get_stepped_graph_coverage(
    G_intermodal,
    gdf_destinations=services,
    weight_type="time_min",
    geometry_type=None,
    weight_value_cutoff=15,
    step=2,
    zone=zone,
)
# Visualize stepped coverage
stepped_cov_voronoi.explore(column='dist', tiles='CartoDB Positron')


In [8]:
stepped_cov_separate = get_stepped_graph_coverage(
    G_intermodal,
    gdf_destinations=services,
    weight_type="time_min",
    geometry_type='separate',
    weight_value_cutoff=15,
    step=2,
)
# Visualize stepped coverage
stepped_cov_separate.explore(column='dist', tiles='CartoDB Positron', vmin=0)


## 6. Radius coverage

Without a graph: Voronoi polygons of the services clipped to 500 m buffers.

In [9]:
radius_cov = get_radius_coverage(gdf_from=services[['name','geometry']], radius=500)
# Visualize radius coverage
radius_cov.explore(column='name', tiles='CartoDB Positron')